# Online Learning Engagement Analysis (DA-015)

Descriptive module and group comparisons using a reproducible synthetic dataset. Engagement is represented by recorded minutes and sessions; this analysis is not causal and does not identify individual students.

## 1. Load and validate data

Run the notebook from the project root after generating the sample CSV if it is not present.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from src.clean_data import aggregate_learner_modules, load_and_clean_data
from src.metrics import activity_score_correlation, module_summary

data_path = Path('data/online_learning_engagement.csv')
if not data_path.exists():
    raise FileNotFoundError('Run `python data/generate_dataset.py` first.')
data = load_and_clean_data(str(data_path))
learner_modules = aggregate_learner_modules(data)
summary = module_summary(learner_modules)
print(f'{data["learner_id"].nunique()} fictional learners; {len(learner_modules)} learner-module records')
summary

## 2. Compare module outcomes

In [ ]:
plot_data = summary.melt(
    id_vars='module',
    value_vars=['completion_rate', 'average_quiz_score'],
    var_name='metric', value_name='percent'
)
sns.set_theme(style='whitegrid')
ax = sns.barplot(data=plot_data, x='module', y='percent', hue='metric')
ax.set(title='Completion and quiz performance by module', xlabel='', ylabel='Percent', ylim=(0, 100))
plt.xticks(rotation=20, ha='right'); plt.tight_layout()

## 3. Explore activity and quiz scores

The Pearson correlation describes association in the observed pairs; it does not establish cause and effect.

In [ ]:
paired = learner_modules.dropna(subset=['activity_minutes', 'quiz_score'])
correlation = activity_score_correlation(learner_modules)
print(f'Available paired records: {len(paired)}; Pearson r: {correlation:.3f}')
sns.scatterplot(data=paired, x='activity_minutes', y='quiz_score', hue='module', alpha=.55)
plt.title('Activity time vs quiz score (descriptive)')
plt.xlabel('Activity minutes per learner-module'); plt.ylabel('Quiz score (0-100)')
plt.tight_layout()

## 4. Check activity data completeness

Missing values remain missing; they are not interpreted as zero activity.

In [ ]:
missing = data.groupby('module', as_index=False).agg(
    rows=('learner_id', 'size'), missing_activity=('activity_missing', 'sum')
)
missing['missing_rate_percent'] = missing['missing_activity'] / missing['rows'] * 100
missing

## Interpretation

Use module/group aggregates to identify questions for course review, not to rank or make high-stakes decisions about individual learners. Validate tracking and combine patterns with course context and learner feedback. The demo data is synthetic.